In [ ]:
"""
=============================================================
FILE 43 — CRITIC GENERATOR PATTERN
=============================================================

CONCEPTS TAUGHT
----------------
1. Critic Generator Architecture
2. AI Self-Evaluation
3. Quality Assurance
4. AI Review Systems
5. Feedback Loops
6. Iterative Refinement
7. AI Governance
8. Autonomous Improvement
9. Critique-Based Optimization
10. Self-Correcting Agents

CORE IDEA
-----------
One agent generates content.
Another agent critiques it.

FLOW
-----
Generator Agent
   ↓
Critic Agent
   ↓
Improved Output

REAL WORLD USE CASES
---------------------
- AI writing systems
- Coding assistants
- Report generation
- Enterprise QA systems
"""

# ============================================================
# STEP 1 — IMPORTS
# ============================================================

import os

from dotenv import load_dotenv

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END

from IPython.display import Image, display

# ============================================================
# STEP 2 — ENV VARIABLES
# ============================================================

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# ============================================================
# STEP 3 — INITIALIZE LLM
# ============================================================

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

# ============================================================
# STEP 4 — DEFINE STATE
# ============================================================

class State(TypedDict):

    topic: str

    generated_content: str

    critic_feedback: str

    improved_content: str

# ============================================================
# STEP 5 — GENERATOR AGENT
# ============================================================

def generator_agent(state: State):

    response = llm.invoke(
        f"""
        Write a detailed article on:

        {state['topic']}
        """
    )

    return {
        "generated_content": response.content
    }

# ============================================================
# STEP 6 — CRITIC AGENT
# ============================================================

def critic_agent(state: State):

    response = llm.invoke(
        f"""
        Critically review the following article.

        Identify:
        - weaknesses
        - missing details
        - unclear reasoning
        - improvement suggestions

        ARTICLE:
        {state['generated_content']}
        """
    )

    return {
        "critic_feedback": response.content
    }

# ============================================================
# STEP 7 — IMPROVEMENT NODE
# ============================================================

def improve_content(state: State):

    response = llm.invoke(
        f"""
        Improve the article using the critic feedback.

        ARTICLE:
        {state['generated_content']}

        FEEDBACK:
        {state['critic_feedback']}
        """
    )

    return {
        "improved_content": response.content
    }

# ============================================================
# STEP 8 — BUILD GRAPH
# ============================================================

builder = StateGraph(State)

builder.add_node(
    "generator_agent",
    generator_agent
)

builder.add_node(
    "critic_agent",
    critic_agent
)

builder.add_node(
    "improve_content",
    improve_content
)

# ============================================================
# STEP 9 — DEFINE EDGES
# ============================================================

builder.add_edge(
    START,
    "generator_agent"
)

builder.add_edge(
    "generator_agent",
    "critic_agent"
)

builder.add_edge(
    "critic_agent",
    "improve_content"
)

builder.add_edge(
    "improve_content",
    END
)

# ============================================================
# STEP 10 — COMPILE GRAPH
# ============================================================

graph = builder.compile()

display(
    Image(
        graph.get_graph().draw_mermaid_png()
    )
)

# ============================================================
# STEP 11 — RUN WORKFLOW
# ============================================================

result = graph.invoke(
    {
        "topic":
        """
        Future of AI in healthcare diagnostics
        """
    }
)

# ============================================================
# STEP 12 — PRINT RESULT
# ============================================================

print("\nIMPROVED ARTICLE\n")
print("=" * 60)

print(result["improved_content"])